In [1]:
import sys
sys.path.append("../src")

In [2]:
from chunking import create_chunks

from embedding import generate_embeddings

from vector_store import build_faiss_store


c:\Users\habtamu.amsalu\Downloads\rag-complaint-chatbot-\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11778.22it/s]


In [3]:
import pandas as pd

df = pd.read_csv("../data/filtered_complaints.csv")

print(df.shape)
df.head()

(478818, 19)


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,cleaned_narrative
0,2025-06-13,Credit card,Store credit card,Getting a credit card,Card opened without my consent or knowledge,A XXXX XXXX card was opened under my name by a...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",TX,78230,Servicemember,Consent provided,Web,2025-06-13,Closed with non-monetary relief,Yes,NaN,14069121,a card was opened under my name by a fraudster...
1,2025-06-13,Checking or savings account,Checking account,Managing an account,Deposits and withdrawals,I made the mistake of using my wellsfargo debi...,Company has responded to the consumer and the ...,WELLS FARGO & COMPANY,ID,83815,NaN,Consent provided,Web,2025-06-13,Closed with explanation,Yes,NaN,14061897,i made the mistake of using my wellsfargo debi...
2,2025-06-12,Credit card,General-purpose credit card or charge card,"Other features, terms, or problems",Other problem,"Dear CFPB, I have a secured credit card with c...",Company has responded to the consumer and the ...,"CITIBANK, N.A.",NY,11220,NaN,Consent provided,Web,2025-06-13,Closed with monetary relief,Yes,NaN,14047085,i have a secured credit card with citibank whi...
3,2025-06-12,Credit card,General-purpose credit card or charge card,Incorrect information on your report,Account information incorrect,I have a Citi rewards cards. The credit balanc...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",IL,60067,NaN,Consent provided,Web,2025-06-12,Closed with explanation,Yes,NaN,14040217,i have a citi rewards cards the credit balance...
4,2025-06-09,Credit card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,b'I am writing to dispute the following charge...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",TX,78413,Older American,Consent provided,Web,2025-06-09,Closed with monetary relief,Yes,NaN,13968411,the following charges on my citi credit card a...


In [4]:
from sampling import create_sample

sample_df = create_sample(
    df,
    sample_size=12000
)

sample_df.shape

(12000, 19)

In [5]:
sample_df.to_csv(
    "../data/sampled_complaints.csv",
    index=False
)

In [6]:
print(sample_df.columns.tolist())

['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID', 'cleaned_narrative']


In [7]:
sample_df.head()


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,cleaned_narrative
0,2025-01-18,"Money transfer, virtual currency, or money ser...",Domestic (US) money transfer,Other transaction problem,NaN,I am filling a complaint against Cash App ( Bl...,NaN,"Block, Inc.",CT,06614,NaN,Consent provided,Web,2025-01-18,Closed with explanation,Yes,NaN,11647396,i am filling a complaint against cash app bloc...
1,2024-06-27,Checking or savings account,Checking account,Managing an account,Deposits and withdrawals,I deposited a two party checks 6 total in the ...,NaN,JPMORGAN CHASE & CO.,IN,471XX,"Older American, Servicemember",Consent provided,Web,2024-06-27,Closed with explanation,Yes,NaN,9346988,i deposited a two party checks total in the am...
2,2023-06-21,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a credit reporting company's inve...,Was not notified of investigation status or re...,It is highly unjustifiable to have late paymen...,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,CA,90703,NaN,Consent provided,Web,2023-06-21,Closed with explanation,Yes,NaN,7150895,it is highly unjustifiable to have late paymen...
3,2022-10-04,Checking or savings account,Checking account,Managing an account,Problem using a debit or ATM card,I disputed various transactions with Capital O...,NaN,CAPITAL ONE FINANCIAL CORPORATION,CA,91754,NaN,Consent provided,Web,2022-10-04,Closed with explanation,Yes,NaN,6045675,i disputed various transactions with capital o...
4,2020-09-27,Credit card or prepaid card,General-purpose credit card or charge card,Closing your account,Company closed your account,I have had a chase credit card for 13 years. T...,NaN,JPMORGAN CHASE & CO.,NJ,07410,NaN,Consent provided,Web,2020-09-27,Closed with explanation,Yes,NaN,3868727,i have had a chase credit card for years the o...


In [8]:
# Check types
print(sample_df[["Complaint ID", "Product", "cleaned_narrative"]].dtypes)

Complaint ID         int64
Product                str
cleaned_narrative      str
dtype: object


In [9]:
# Check how many nulls are in each column
print(sample_df[["Complaint ID", "Product", "cleaned_narrative"]].isnull().sum())

Complaint ID         0
Product              0
cleaned_narrative    1
dtype: int64


In [10]:
sample_df = sample_df.dropna(subset=["cleaned_narrative"])

In [11]:
chunks_df = create_chunks(sample_df)

chunks_df.head()

,complaint_id,product,chunk_id,chunk_text
0,11647396,"Money transfer, virtual currency, or money ser...",0,i am filling a complaint against cash app bloc...
1,11647396,"Money transfer, virtual currency, or money ser...",1,did not comply with error resolution requireme...
2,9346988,Checking or savings account,0,i deposited a two party checks total in the am...
3,9346988,Checking or savings account,1,the that i had left in the bank and my ss that...
4,7150895,Credit card or prepaid card,0,it is highly unjustifiable to have late paymen...


In [12]:
embeddings = generate_embeddings(
    chunks_df["chunk_text"].tolist()
)

Batches: 100%|██████████| 1074/1074 [12:11<00:00,  1.47it/s]


In [13]:

build_faiss_store(
    embeddings,
    chunks_df
)

FAISS store created successfully
Vectors stored: 34338
Dimension: 384
c:\Users\habtamu.amsalu\Downloads\rag-complaint-chatbot-\vector_store\faiss_index.bin
c:\Users\habtamu.amsalu\Downloads\rag-complaint-chatbot-\vector_store\metadata.pkl


In [14]:
import os

print(os.path.exists("../vector_store/faiss_index.bin"))
print(os.path.exists("../vector_store/metadata.pkl"))

True
True


In [15]:
print(chunks_df.columns.tolist())

['complaint_id', 'product', 'chunk_id', 'chunk_text']


In [16]:
import os

os.makedirs("../vector_store", exist_ok=True)

print(os.path.abspath("../vector_store"))
print(os.path.exists("../vector_store"))

c:\Users\habtamu.amsalu\Downloads\rag-complaint-chatbot-\vector_store
True
